# BARAM 2026 — FICR 개선 2후보 실험 (구간별 nudge · FICR-지향 점추정)

v6(LB 0.6292, 212위) 이후. FICR 갭(0.389 vs 상위 0.42~0.455)이 최대 지렛대.

- **C0 (기준선=v6)**: blend + 전역 보수 nudge(scale≤1.05, shift≤2%)
- **C1**: blend + **예측CF 구간별**(0-25/25-50/50-75/75-100%) 보수 nudge — 구간 기준은 예측값(테스트에서도 가용, 누설 없음). 구간 유효표본<300이면 전역 파라미터로 폴백
- **C2**: **FICR-지향 점추정** — fold-train에서 LGBM quantile(τ=0.1..0.9) 9개 학습 → 분포를 blend 예측 중심으로 재중심화 → 시간별로 공식 점수 기여 J(p)=E[1{y≥10%cap}·(−|y−p|/cap + y·U(|y−p|/cap)/(4·ȳ))] 최대 점 선택

판정: **2023·2024 두 폴드 모두 C0 이상**만 채택 (v5 교훈).
v6 실측 캘리브레이션: 폴드 2024 기대 0.6294 ↔ LB 0.6292 — 이 하네스는 신뢰 가능.

## 0. 설정 & 폴드 blend 예측 + OOF (v6와 동일 파이프라인, MLP 시드3)

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1]; BLEND=0.7
MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
GROUPS=(1,2,3)
FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def lgbm_q(alpha): return lgb.LGBMRegressor(objective="quantile",alpha=alpha,n_estimators=500,
    learning_rate=0.05,num_leaves=63,min_child_samples=60,subsample=0.8,subsample_freq=1,
    colsample_bytree=0.7,reg_lambda=1.0,random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)

class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_one(pool_tr,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[FEATS].mean(); sd=pool_tr[FEATS].std()+1e-8
    X=((pool_tr[FEATS]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV); gt=torch.tensor(gid,device=DEV)
    net=MLP(len(FEATS),**{k:MLP_CFG[k] for k in ("h","depth","drop","emb")}).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=MLP_CFG["lr"],weight_decay=MLP_CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),MLP_CFG["bs"]):
            b=torch.tensor(perm[i:i+MLP_CFG["bs"]],device=DEV)
            opt.zero_grad(); ((net(Xt[b],gt[b])-yt[b]).abs().mean()).backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs().mean().item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[FEATS]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

def blend_predict(tr_frames):
    pool=[]
    for g,(tr2,_) in tr_frames.items():
        p=tr2[FEATS+["kst_dtm"]].copy(); p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1; pool.append(p)
    pool=pd.concat(pool,ignore_index=True)
    acc={g:[] for g in tr_frames}
    for sd_ in SEEDS:
        net,scaler=train_one(pool,sd_)
        for g,(_,va2) in tr_frames.items():
            acc[g].append(predict_one(net,scaler,va2,g,W.CAP[g]))
    out={}
    for g,(tr2,va2) in tr_frames.items():
        cap=W.CAP[g]; tgt=TGT[g]
        gp=np.clip(0.5*(lgbm().fit(tr2[FEATS],tr2[tgt]).predict(va2[FEATS])
             +histgbm().fit(tr2[FEATS].to_numpy(),tr2[tgt].to_numpy()).predict(va2[FEATS].to_numpy())),0,cap)
        out[g]=np.clip((1-BLEND)*gp+BLEND*np.mean(acc[g],axis=0),0,cap)
    return out

FOLDS={2023:[2022],2024:[2022,2023]}
DATA={}
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=dict(tr2=W.with_pc(tr,iso),va2=W.with_pc(va,iso))
    pred=blend_predict({g:(e["tr2"],e["va2"]) for g,e in ent.items()})
    for g in ent: ent[g]["pred"]=pred[g]
    for g,e in ent.items():
        tr2=e["tr2"]; oof=np.full(len(tr2),np.nan); yrs=sorted(tr2.kst_dtm.dt.year.unique())
        if len(yrs)>=2:
            for ty in yrs:
                m_in=tr2.kst_dtm.dt.year!=ty; m_out=(tr2.kst_dtm.dt.year==ty).to_numpy()
                oof[m_out]=blend_predict({g:(tr2[m_in],tr2[tr2.kst_dtm.dt.year==ty])})[g]
        else:
            n=len(tr2); cut=int(n*0.7)
            oof[cut:]=blend_predict({g:(tr2.iloc[:cut],tr2.iloc[cut:])})[g]
        e["oof"]=oof
    DATA[vy]=ent
    print(f"fold →{vy} ready")

fold →2023 ready


fold →2024 ready


## 1. C0/C1: 전역 vs 구간별 보수 nudge

In [2]:
def cons_nudge_fit(yt,yp,cap,smax=1.05,shmax=0.02):
    best=(1.0,0.0); bf=-1
    for s in np.linspace(2-smax,smax,21):
        for sh in np.linspace(-shmax,shmax,21)*cap:
            f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
            if f>bf: bf=f; best=(s,sh)
    return best

SEG_EDGES=[0,0.25,0.5,0.75,1.01]
def seg_of(pred,cap): return np.digitize(pred/cap,SEG_EDGES)-1

results={}
for vy,ent in DATA.items():
    for g,e in ent.items():
        tgt=TGT[g]; cap=W.CAP[g]; tr2=e["tr2"]; va2=e["va2"]
        m=np.isfinite(e["oof"]); yt_o=tr2[tgt].to_numpy()[m]; yp_o=e["oof"][m]
        # C0 전역
        s0,sh0=cons_nudge_fit(yt_o,yp_o,cap)
        p_c0=np.clip(e["pred"]*s0+sh0,0,cap)
        # C1 구간별 (예측 기준 구간, 폴백=전역)
        segs_o=seg_of(yp_o,cap); segs_v=seg_of(e["pred"],cap)
        p_c1=np.empty_like(e["pred"])
        for si in range(4):
            mo=segs_o==si; mv=segs_v==si
            if not mv.any(): continue
            # FICR은 유효구간(실측>=10%)만 채점되므로 fit 표본도 그 기준으로 계산됨(group_scores 내부)
            if mo.sum()>=300:
                s,sh=cons_nudge_fit(yt_o[mo],yp_o[mo],cap)
            else:
                s,sh=s0,sh0
            p_c1[mv]=np.clip(e["pred"][mv]*s+sh,0,cap)
        e["C0"]=p_c0; e["C1"]=p_c1
    print(f"fold →{vy}: C0/C1 done")

fold →2023: C0/C1 done


fold →2024: C0/C1 done


## 2. C2: FICR-지향 점추정 (quantile 분포 → 기대점수 최대 점)

In [3]:
QS=np.arange(0.1,0.91,0.1)
def ficr_point(tr2,va2,tgt,cap,anchor):
    """LGBM quantile 분포를 anchor(blend 예측) 중심으로 재중심화 → J(p) 최대 점."""
    Xtr=tr2[FEATS]; ytr=tr2[tgt]
    qpred=np.stack([np.clip(lgbm_q(a).fit(Xtr,ytr).predict(va2[FEATS]),0,cap) for a in QS],axis=1)
    qpred=np.sort(qpred,axis=1)
    med=qpred[:,len(QS)//2]
    dist=np.clip(qpred-med[:,None]+anchor[:,None],0,cap)   # 재중심화된 분포 샘플 (N,9)
    ybar=max(tr2[tgt][tr2[tgt]>=0.1*cap].mean(),1e-6)      # 학습기간 유효구간 평균(정규화 상수)
    # 후보점: anchor 주변 그리드 (분포 범위 내 21점)
    lo=dist.min(axis=1); hi=dist.max(axis=1)
    grid=lo[:,None]+(hi-lo)[:,None]*np.linspace(0,1,21)[None,:]      # (N,21)
    er=np.abs(dist[:,None,:]-grid[:,:,None])/cap                      # (N,21,9)
    price=np.select([er<=0.06,er<=0.08],[4.0,3.0],default=0.0)
    validm=(dist>=0.1*cap)[:,None,:]                                  # 유효구간만 채점
    J=(validm*(-er + dist[:,None,:]*price/(4*ybar*cap))).mean(axis=2) # (N,21)
    return np.take_along_axis(grid,J.argmax(axis=1)[:,None],axis=1).squeeze(1)

for vy,ent in DATA.items():
    for g,e in ent.items():
        e["C2"]=np.clip(ficr_point(e["tr2"],e["va2"],TGT[g],W.CAP[g],e["pred"]),0,W.CAP[g])
    print(f"fold →{vy}: C2 done")

fold →2023: C2 done


fold →2024: C2 done


## 3. 판정 (두 폴드 공식 총점)

In [4]:
def fold_total(vy,key):
    nm=[]; fi=[]
    for g,e in DATA[vy].items():
        yt=e["va2"][TGT[g]].to_numpy()
        a,b=group_scores(yt,e[key],W.CAP[g]); nm.append(a); fi.append(b)
    return 0.5*(1-np.mean(nm))+0.5*np.mean(fi)

res={}
for key in ["C0","C1","C2"]:
    t23,t24=fold_total(2023,key),fold_total(2024,key)
    res[key]=(t23,t24)
    print(f"{key}: 2023={t23:.4f}  2024={t24:.4f}  min={min(t23,t24):.4f}  mean={(t23+t24)/2:.4f}")

base23,base24=res["C0"]
adopt=[k for k in ["C1","C2"] if res[k][0]>=base23 and res[k][1]>=base24]
winner=max(adopt,key=lambda k:min(res[k])) if adopt else "C0"
print(f"\n판정: {winner} " + ("채택" if winner!="C0" else "(개선 없음 → v6 유지)"))

summary=dict(totals={k:{"2023":round(res[k][0],4),"2024":round(res[k][1],4)} for k in res},
             winner=winner,
             note="C0=v6 전역 보수nudge / C1=구간별 / C2=quantile FICR-지향 점추정")
json.dump(summary,open("submission/ver_6/ficr2_summary.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

C0: 2023=0.6117  2024=0.6297  min=0.6117  mean=0.6207
C1: 2023=0.6099  2024=0.6301  min=0.6099  mean=0.6200
C2: 2023=0.6048  2024=0.6154  min=0.6048  mean=0.6101

판정: C0 (개선 없음 → v6 유지)
{
  "totals": {
    "C0": {
      "2023": 0.6117,
      "2024": 0.6297
    },
    "C1": {
      "2023": 0.6099,
      "2024": 0.6301
    },
    "C2": {
      "2023": 0.6048,
      "2024": 0.6154
    }
  },
  "winner": "C0",
  "note": "C0=v6 전역 보수nudge / C1=구간별 / C2=quantile FICR-지향 점추정"
}


## 4. 결론
- 채택된 후보(C1/C2)가 있으면 PIPELINE_FINAL을 v7로 갱신해 제출 파일 생성 (별도 단계).
- 캘리브레이션 기준: 이 하네스의 2024 폴드 값 ≈ LB (v6에서 확인: 0.6294 ↔ 0.6292).